# TC vortex gravity-wave model output

This notebook plots the outward power-flux time series and animates one snapshot variable. Run the model before executing these cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

# This works when Jupyter starts in either the repository root or plots/.
cwd = Path.cwd().resolve()
repo_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "src" / "config.yaml").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

output_dir = repo_root / "outputs"
snapshot_dir = output_dir / "snapshots"
print(f"Reading model output from {output_dir}")

## 1. Power-flux time series

In [ ]:
power_path = output_dir / "power_flux_timeseries.csv"
if not power_path.exists():
    raise FileNotFoundError(f"Run the model first; missing {power_path}")

power_data = np.atleast_1d(np.genfromtxt(power_path, delimiter=",", names=True))
time_hours = power_data["time_s"] / 3600.0

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(time_hours, power_data["power_W"], color="tab:blue", linewidth=2)
ax.axhline(0.0, color="0.35", linewidth=0.8)
ax.set(
    xlabel="Simulation time (h)",
    ylabel="Outward power flux (W)",
    title="Perturbation power flux through the diagnostic circle",
)
ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Snapshot animation

Choose `u`, `v`, or `h` below. The snapshots contain total model fields.

In [ ]:
variable = "u"  # Choose "u", "v", or "h".
frame_interval_ms = 150

if variable not in {"u", "v", "h"}:
    raise ValueError('variable must be one of "u", "v", or "h"')

snapshot_paths = sorted(snapshot_dir.glob("snapshot_*.npz"))
if not snapshot_paths:
    raise FileNotFoundError(f"Run the model first; no snapshots found in {snapshot_dir}")

# Scan one frame at a time to get a fixed color scale without retaining all fields.
field_min, field_max = np.inf, -np.inf
for path in snapshot_paths:
    with np.load(path) as snapshot:
        field = snapshot[variable]
        field_min = min(field_min, float(field.min()))
        field_max = max(field_max, float(field.max()))

if variable in {"u", "v"}:
    limit = max(abs(field_min), abs(field_max), np.finfo(float).eps)
    vmin, vmax, cmap = -limit, limit, "RdBu_r"
else:
    vmin, vmax, cmap = field_min, field_max, "viridis"
    if np.isclose(vmin, vmax):
        vmax = vmin + 1.0

with np.load(snapshot_paths[0]) as first:
    x_km = first["x"] / 1000.0
    y_km = first["y"] / 1000.0
    first_field = first[variable]
    first_time_h = float(first["t"].item()) / 3600.0

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(
    first_field,
    origin="lower",
    extent=(x_km[0], x_km[-1], y_km[0], y_km[-1]),
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest",
)
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label({"u": "u (m s⁻¹)", "v": "v (m s⁻¹)", "h": "h (m)"}[variable])
ax.set(xlabel="x (km)", ylabel="y (km)")
title = ax.set_title(f"{variable} at t = {first_time_h:.1f} h")
fig.tight_layout()

def update(frame_index):
    with np.load(snapshot_paths[frame_index]) as snapshot:
        image.set_data(snapshot[variable])
        time_h = float(snapshot["t"].item()) / 3600.0
    title.set_text(f"{variable} at t = {time_h:.1f} h")
    return image, title

model_animation = FuncAnimation(
    fig,
    update,
    frames=len(snapshot_paths),
    interval=frame_interval_ms,
    blit=False,
)
plt.close(fig)
HTML(model_animation.to_jshtml())